# 🚀 AGAR-RL V8 : Authentic Competitive Physics (16-Cell Virus Mechanics, Wall-Bounce Feeding & Subcell Threat Perception)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Albin0903/agario/blob/main/notebooks/train_colab.ipynb)

**Entraînement de haute performance à pleine puissance (GPU L4 / A100) avec auto-sauvegarde Google Drive.**

### 🎯 Nouveautés Majeures de la V8 (Authentic Agar.io Competitive Physics) :
1. **Mécanique Authentique des 16 Sous-Cellules & Absorption de Virus** : Plafond officiel à 16 cellules (`max_subcells: 16`) et seuil de split minimal à 36 (`min_split_mass: 36.0`). Si $< 16$ sous-cellules, percuter un virus ($> 130$ masse) fait éclater la cellule. Dès que le joueur atteint le plafond de 16 cellules, percuter un virus l'absorbe sans éclater et octroie $+100$ de masse (*virus farming* compétitif standard). Les cellules $\le 130$ se dissimulent sous les virus en sécurité.
2. **Rebond Mural des Feeds (*Wall-Bounce Feeding*)** : Les projections de masse ($W$) rebondissent sur les bords de l'arène avec inversion de vitesse (`em.vx = -0.75 * em.vx`), permettant aux sous-cellules de transférer leur masse à leur cellule principale en toute sécurité sans concéder de masse aux ennemis.
3. **Tir de Virus par Éjection de Masse ($W$)** : Alimenter un virus jusqu'à 140 de masse déclenche un tir balistique sur 350 unités qui fait exploser les gros adversaires dans la ligne de mire.
4. **Perception Proies vs Prédateurs au Niveau Sous-Cellulaire** : Élimination totale du faux sentiment de supériorité : une cible n'est classée comme **Proie** que si notre plus grosse sous-cellule peut effectivement l'avaler (`other_cell.mass * 1.1 <= max_subcell_mass`). Une cible est classée comme **Prédateur** dès qu'elle menace au moins un de nos morceaux (`other_cell.mass >= 1.1 * min_subcell_mass`).
5. **Zéro Biais Artificiel de Reward** : La fonction de reward reste pure et minimaliste (croissance de masse réelle + élimination). L'apprentissage des tactiques est organique via les affordances physiques de l'arène.
6. **Reprise Transparente Hiérarchique depuis V7 / V6** : Détection automatique : `agario_rl_backup_v8` en priorité, sinon reprise immédiate du dernier palier de `agario_rl_backup_v7` ou `agario_rl_backup_v6` !
7. **Sauvegarde Continue V8** : Dossier de backup Google Drive `/content/drive/MyDrive/agario_rl_backup_v8`, export ONNX `model_v8.onnx`, et replay `eval_match_v8.mp4`.

## 0. Montage Google Drive & Détection GPU L4
Tous les checkpoints, les replays HD et les modèles ONNX seront automatiquement sauvegardés sur votre Drive dans le dossier `agario_rl_backup_v8`.

In [ ]:
# 1. Montage sécurisé de Google Drive
import os, sys, time, torch

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    pass

DRIVE_BACKUP_DIR = '/content/drive/MyDrive/agario_rl_backup_v8'
PREV_BACKUP_V7 = '/content/drive/MyDrive/agario_rl_backup_v7'
PREV_BACKUP_V6 = '/content/drive/MyDrive/agario_rl_backup_v6'
PREV_BACKUP_V5 = '/content/drive/MyDrive/agario_rl_backup_v5'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# 2. Vérification du matériel accéléré (GPU L4 / A100 recommandé)
print('=' * 65)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'🚀 Accélération GPU Détectée : {gpu_name} ({vram:.1f} Go VRAM)')
    print('⚡ Configuration optimale : 16 environnements parallèles + Numba JIT + Batch 512')
else:
    print('⚠️ Aucun GPU détecté. Activez un GPU dans : Exécution > Modifier le type d\'exécution')
print(f'📁 Dossier Google Drive V8 synchronisé : {DRIVE_BACKUP_DIR}')
print(f'📁 Dossier V7 précédent disponible : {PREV_BACKUP_V7} (Existe: {os.path.exists(PREV_BACKUP_V7)})')
print(f'📁 Dossier V6 précédent disponible : {PREV_BACKUP_V6} (Existe: {os.path.exists(PREV_BACKUP_V6)})')
print('=' * 65)

## 1. Synchronisation du Code GitHub & Installation des Dépendances

In [ ]:
import os

# 1. Récupération propre des dernières modifications ou clonage
if os.path.exists('.git'):
    print('🔄 Synchronisation avec GitHub main...')
    !git fetch origin main
    !git reset --hard origin/main
elif os.path.exists('agario/.git'):
    print('🔄 Déplacement dans agario et synchronisation avec GitHub main...')
    %cd agario
    !git fetch origin main
    !git reset --hard origin/main
else:
    print('🌐 Clonage propre du repo...')
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

# 2. Configuration du PYTHONPATH et installation des dépendances Farama Gymnasium
os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip uninstall -y -q gym 2>/dev/null || true
!pip install -q -r requirements.txt tensorboard
!apt-get install -qq -y ffmpeg
print('✅ Environnement et dépendances installés avec succès.')

## 2. Validation Pré-Vol : Suite Complète de 34 Tests Unitaires
Vérification complète de la physique du moteur, du remerge magnétique, des récompenses de traque et de l'espace d'observation log-ratio.

In [ ]:
# Exécute tous les tests du moteur physique, du remerge et des récompenses Farama
!python -m pytest -v

## 3. Monitoring TensorBoard (Optionnel)

In [ ]:
import os
os.makedirs('logs/tensorboard', exist_ok=True)
try:
    %load_ext tensorboard
    %tensorboard --logdir logs/tensorboard
except Exception as e:
    print(f'Note TensorBoard : {e}')

## 4. Entraînement Haute Performance V8 (Authentic Competitive Agar.io)
- **Reprise Optimale depuis V7 / V6** : Détection automatique du plus récent checkpoint dans `agario_rl_backup_v7` (ou `v6`) pour continuer l'entraînement avec la physique authentique des 16 cellules et la perception sous-cellulaire !
- **Sauvegarde Continue V8** : Checkpoints automatiques tous les 250 000 pas dans `agario_rl_backup_v8`.

In [ ]:
# 🚀 Configuration de Reprise & Lancement V8
import os, glob, re

V8_DIR = '/content/drive/MyDrive/agario_rl_backup_v8'
V7_DIR = '/content/drive/MyDrive/agario_rl_backup_v7'
V6_DIR = '/content/drive/MyDrive/agario_rl_backup_v6'
V5_DIR = '/content/drive/MyDrive/agario_rl_backup_v5'
os.makedirs(V8_DIR, exist_ok=True)

def extract_step(path):
    fname = os.path.basename(path)
    if 'final' in fname:
        return 999_999_999
    m = re.search(r'step_(\d+)', fname)
    return int(m.group(1)) if m else 0

# Recherche hiérarchique du meilleur checkpoint : V8 d'abord, puis V7, puis V6, puis V5
def find_best_checkpoint(dir_path):
    if not os.path.exists(dir_path):
        return None
    zips = glob.glob(os.path.join(dir_path, '*.zip'))
    valid = [z for z in zips if os.path.getsize(z) > 1000 and not os.path.basename(z).startswith('._') and 'bc_pretrained' not in z]
    valid.sort(key=extract_step, reverse=True)
    latest = os.path.join(dir_path, 'ppo_latest.zip')
    if valid:
        return valid[0]
    if os.path.exists(latest):
        return latest
    return None

chosen_checkpoint = find_best_checkpoint(V8_DIR) or find_best_checkpoint(V7_DIR) or find_best_checkpoint(V6_DIR) or find_best_checkpoint(V5_DIR)

if chosen_checkpoint:
    resume_flag = f'--resume "{chosen_checkpoint}"'
    step_num = extract_step(chosen_checkpoint)
    print('=' * 75)
    print(f'🎯 Checkpoint source détecté pour reprise : {chosen_checkpoint}')
    print(f'📊 Palier détecté : {step_num:,} steps' if step_num < 999_999_999 else '📊 Palier : FINAL')
    print('=' * 75)
else:
    resume_flag = '--resume auto'
    print('⚠️ Aucun checkpoint préalable trouvé, démarrage d\'un nouvel entraînement.')

!python src/training/train_colab.py \
    --n-envs 16 \
    --total-timesteps 20000000 \
    --pool-interval 250000 \
    --backup-dir {V8_DIR} \
    {resume_flag} \
    --device auto

## 5. Inspection Diagnostique de la Politique & Réflexes Tactiques
Sonde le réseau de neurones sur des scénarios synthétiques contrôlés : réponse à la nourriture, esquive des prédateurs mortels vs calme face aux rivaux inoffensifs, et propension à attaquer les proies en zone de frappe.

In [ ]:
import os, glob, re

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

candidates = glob.glob('/content/drive/MyDrive/agario_rl_backup_v8/*.zip') + \
             glob.glob('/content/drive/MyDrive/agario_rl_backup_v7/*.zip') + \
             glob.glob('/content/drive/MyDrive/agario_rl_backup_v6/*.zip') + \
             glob.glob('checkpoints/ppo/*.zip')

candidates = [c for c in candidates if os.path.getsize(c) > 1000 and not os.path.basename(c).startswith('._') and 'bc_pretrained' not in c]
candidates.sort(key=extract_step, reverse=True)
target_inspect = candidates[0] if candidates else 'checkpoints/ppo/ppo_latest.zip'

print('=' * 75)
print(f'🔬 Inspection Diagnostique du Modèle : {target_inspect}')
print('=' * 75)

!python src/analysis/inspect_policy.py --model "{target_inspect}"

## 6. Enregistrement Automatique du Match Replay HD & Visualisation Directe
Génère une vidéo HD de 80 secondes (2400 steps @ 30 FPS) avec affichage tête haute (HUD), vecteurs de décision et radar.

In [ ]:
import os, glob, re
from IPython.display import HTML, display
from base64 import b64encode

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

candidates = glob.glob('/content/drive/MyDrive/agario_rl_backup_v8/*.zip') + \
             glob.glob('/content/drive/MyDrive/agario_rl_backup_v7/*.zip') + \
             glob.glob('/content/drive/MyDrive/agario_rl_backup_v6/*.zip') + \
             glob.glob('checkpoints/ppo/*.zip')

valid_cands = [c for c in candidates if os.path.getsize(c) > 1000 and not os.path.basename(c).startswith('._') and 'bc_pretrained' not in c]
valid_cands.sort(key=extract_step, reverse=True)
target_model = valid_cands[0] if valid_cands else 'checkpoints/ppo/ppo_latest.zip'
step_count = extract_step(target_model)

print('=' * 75)
print(f'🎬 Modèle sélectionné pour le Replay HD : {target_model}')
print(f'📊 Palier : {step_count:,} steps')
print('=' * 75)

os.makedirs('recordings', exist_ok=True)
!python src/inference/record_match.py \
    --model "{target_model}" \
    --output recordings/eval_match_v8.mp4 \
    --steps 2400

if os.path.exists('recordings/eval_match_v8.mp4') and os.path.exists('/content/drive/MyDrive/agario_rl_backup_v8'):
    !cp recordings/eval_match_v8.mp4 /content/drive/MyDrive/agario_rl_backup_v8/eval_match_v8.mp4
    print('📁 Replay HD copié sur Google Drive dans : agario_rl_backup_v8/eval_match_v8.mp4')

video_path = 'recordings/eval_match_v8.mp4'
if os.path.exists(video_path):
    mp4_bytes = open(video_path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <video width="850" height="480" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
    print(f'Taille de la vidéo : {os.path.getsize(video_path) / 1_000_000:.1f} Mo')
else:
    print('⚠️ Vidéo non trouvée.')

## 7. Exportation Universelle vers ONNX & Benchmark de Latence
Convertit le réseau de neurones PyTorch au standard ONNX ultra-rapide (< 0.02 ms de latence CPU).

In [ ]:
import os, glob, re

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

candidates = glob.glob('/content/drive/MyDrive/agario_rl_backup_v8/*.zip') + \
             glob.glob('/content/drive/MyDrive/agario_rl_backup_v7/*.zip') + \
             glob.glob('/content/drive/MyDrive/agario_rl_backup_v6/*.zip') + \
             glob.glob('checkpoints/ppo/*.zip')

valid_cands = [c for c in candidates if os.path.getsize(c) > 1000 and not os.path.basename(c).startswith('._') and 'bc_pretrained' not in c]
valid_cands.sort(key=extract_step, reverse=True)
best_model = valid_cands[0] if valid_cands else None

if best_model and os.path.exists(best_model):
    print(f'Modèle sélectionné pour l\'export : {best_model}')
    os.makedirs('models', exist_ok=True)
    !python src/inference/export_onnx.py --model "{best_model}" --output models/model_v8.onnx
    if os.path.exists('/content/drive/MyDrive/agario_rl_backup_v8'):
        !cp models/model_v8.onnx /content/drive/MyDrive/agario_rl_backup_v8/model_v8.onnx
        print('📁 Modèle ONNX sauvegardé sur Drive : agario_rl_backup_v8/model_v8.onnx')
else:
    print('⚠️ Aucun checkpoint trouvé pour l\'export ONNX.')